# Lab. 7 - K-means and the Lloyd algorithm

In this lab we consider the **unsupervised learning** problem of **clustering**. We will look at the K-means approach, and in particular to one of its most widely known implementations: the Lloyd algorithm.

As usual, we start by importing the relevant libraries and by defining the `mixGauss` function to generate our dataset.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def mixGauss(means, sigmas, n, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    means = np.array(means)
    sigmas = np.array(sigmas)

    d = means.shape[1]
    num_classes = sigmas.size
    data = np.full((n * num_classes, d), np.inf)
    labels = np.zeros(n * num_classes)

    for idx in range(len(sigmas)):
        data[idx * n:(idx + 1) * n] = rng.multivariate_normal(
            mean=means[idx], cov=np.eye(d) * sigmas[idx] ** 2, size=n)
        labels[idx * n:(idx + 1) * n] = idx

    if(num_classes == 2):
        labels[labels==0] = -1

    return data, labels

## Implementing the Lloyd Algorithm

The Lloyd algorithm is the standard algorithm for implementing k-means. It is based on two steps:
 1. Assigning each point to a cluster,
 2. Updating the cluster centers.

Loop until the cluster centers (and label assignments) have converged.

The function below takes as input the following arguments:
 - `X`: the input dataset,
 - `centers`: the initialisation of the cluster centers,
 - `maxiter` the maximum number of iterations of the algorithm.

The function returns:
 - `X_centers`: the ordered labels assigned to the data points `X`,
 - `centers_history`: the history of the cluster centers at each iteration.

In [ ]:
# Compute the euclidean distance between each pair of the two sets of points X1 and X2
def all_distances(X1, X2):
    ...

    return distances


def lloyd(X, centers, max_iter):
    """
    K-means clustering using Lloyd's algorithm.

    Args:
        X (np.ndarray): Data points of shape (n_samples, n_features).
        centers (np.ndarray): Initial centers of shape (n_clusters, n_features).
        max_iter (int): Maximum number of iterations.

    Returns:
        cluster_assignments (np.ndarray): Index of assigned cluster for each data point.
        centers_history (list of np.ndarray): History of centers after each iteration.
    """
    # Extract number of input points and dimensions from the shape of input X
    n, d = X.shape

    # Extract number of centers k
    k = ...

    # Initialize history of the centers coordinates with the initial positions
    centers_history = [centers.copy()]

    # For every iteration
    for i in range(max_iter):
        # Compute the euclidean distance between each cluster center and each point
        dist = all_distances(...)

        # Assign point to clusters:
        # For each point, find the closest center in terms of euclidean distance
        cluster_assignments = ...

        # Update cluster center
        for c in range(k):
            if len(X[cluster_assignments == c]) > 0:
              centers[c] = ...
            # Q: if a center has no points assigned (i.e. if X[cluster_assignments == c] is empty),
            # what should its coordinates be assigned to?
            else:
              centers[c] = ...

        # Add the new centers positions to the history
        centers_history.append(centers.copy())

    return cluster_assignments, centers_history

Q: With the function defined in this way, we have to manually set the number of iterations of the algorithm. Can you think of a better method of stopping the algorithm that does not require us to set the total number of iterations a priori?

Let us try the Lloyd algorithm for K-means method. First, we create a synthetic dataset using the `mixGauss` function **with three or more classes**.

We also have to give the initial coordinates of the centers to the algorithm. As a first strategy, we can initialise them randomly. As we will see later, there are other (better) initailisation strategies.

We can experiment with different datasets:
 - A dataset where the classes are very well separated, for which we expect the Lloyd algorithm to infer the classes easily,
 - A dataset where the classes have overlap. In this case the Lloyd algorithm shouldn't be able to distinguish the points of overlap. What do you think will happen?

In [ ]:
# Build a dataset with at least 3 classes in 2 dimensions, and plot it
X, Y = mixGauss(
    means=[[0., 0.], [0.65, 0.55], [-0.25, -0.35]],
    sigmas=[0.21, 0.18, 0.1],
    n=100, rng=np.random.default_rng(8)
)
plt.scatter(X[:,0], X[:,1], s=70, c=Y, alpha=0.8);

In [ ]:
# Define number of centers
k = ...

# Randomly initialize the centers
# Careful: the bounds within which we randomly pick the centers must be the same of the dataset
centers0 = np.random.uniform(low=..., high=..., size=...)

# Set the number of iterations
n_iters = ...

# Run the Lloyd's algorithm
cluster_assignments, centers_history = ...

In [ ]:
# Visualize
_, axes = plt.subplots(1, 2, figsize=(12, 5))

# Visualize original classses
axes[0].scatter(X[:,0], X[:,1], s=70, c=Y, alpha=0.8)
axes[0].set_title('Dataset')

# Visualize the final clusters
centers = centers_history[-1]
axes[1].scatter(X[:,0], X[:,1], s=70, c=cluster_assignments, alpha=0.8)
# Plot the trajectory of centers during the iterations
styles = ['o-r', 'd-g', '+-b', 's-c', '*-y']
for c in range(k):
  axes[1].plot(
    [centers_history[i][c][0] for i in range(n_iters)],
    [centers_history[i][c][1] for i in range(n_iters)],
    styles[c]
  )

# Plot the final positions of centers
axes[1].plot(centers[:,0], centers[:,1], '*k', ms=10, zorder=10)
axes[1].set_title(f'Clusters [n_iters={n_iters}]');

Q: Try to run the algorithm multile times, with different initialisations of the centers. Do you obtain the same clusters every time? Why?

Q: Try using the *wrong* number of clusters `k` (wrong with respect to the number of classes you used to generate the data). What happens?

## Using a library: Scikit-learn

In this part, we see how we can approach the problem of clustering with the K-means method by using an external library: [Scikit-learn](https://scikit-learn.org/stable/).

Here is an example to obtain a very similar implementation of the K-means method as the one we wrote from scratch above.

In [ ]:
# From sklearn library import KMeans and metrics classes
from sklearn.cluster import KMeans
from sklearn import metrics

# Define range of the number of clusters to check
k_range = range(2, 21)

# Initialize empty list to store the score of the algorithm for every k
scores = []

# For every k
for k in k_range:
    # Run the algorithm with k clusters and random initial centers on the dataset X
    km = KMeans(n_clusters=k, random_state=1, init='random', n_init=1, max_iter=100)
    km.fit(X)

    # Evaluate the score of the clustering and append it to the list
    scores.append(metrics.silhouette_score(X, km.labels_))

# Plot results
plt.plot(k_range, scores)
plt.grid(True)
plt.xlabel('Number of clusters')
plt.ylabel('Score')
plt.xticks(k_range);

Q: Look at the documentation of the KMeans class; can you change the initialisation strategy? What does the `n_init` parameter do? Try to change both parameters and comment the results.

## A better initialization strategy: K-means++

Initializing the cluster centers at random, sometimes makes the algorithm converge to a sub-optimal clustering.

The K-means++ algorithm is a greedy **initialization strategy** to initialize the centers, which can then be passed to the Lloyd algorithm.

K-means++ works as follows:
* Randomly pick first center.
* For each point, maintain the distance to the closest center so far.
* Sample a new center with probability proportional to squared distance.
* Repeat until k centers are chosen.

Let us implement this strategy in the `kmeanspp` function below.

In [ ]:
def kmeanspp(X, k):
    """
    K-means++ initialization.

    Args:
        X (np.ndarray): Dataset of shape (n_samples, n_features).
        k (int): Number of clusters.

    Returns:
        centers (np.ndarray): Initialized centers of shape (k, n_features).
    """
    # Extract number of input points and dimensions from the shape of input X
    n, d = X.shape
    centers = np.empty((k, d), dtype=X.dtype)

    # Randomly choose the first center
    centers[0] = ...

    # Initialize array of closest distances squared, set all to np.inf
    closest_dist_sq = ...

    for i in range(1, k):
        # Compute squared distances to the latest center
        distances_to_new_center = ...

        # Update the minimum distance to any center
        closest_dist_sq = ...

        # Choose next center with probability proportional to distance squared
        probs = ...
        next_center_idx = np.random.choice(n, p=probs)
        centers[i] = ...

    return centers

Now, let us take a dataset on which random-initialized K-means struggles, and apply K-means++ for the center initialization, and let's check if the new initialization strategy improves the clustering.

In [ ]:
# Try and use K-Means++ for initialization
k = ...
centers0 = kmeanspp(X, k)

# Run the Lloyd algorithm
n_iters = ...
cluster_assignments, centers_history = ...

In [ ]:
# Visualize
_, axes = plt.subplots(1, 2, figsize=(12, 5))
# Visualize original classses
axes[0].scatter(X[:,0], X[:,1], s=70, c=Y, alpha=0.8)
axes[0].set_title('Dataset')

# Visualize the final clusters
centers = centers_history[-1]
axes[1].scatter(X[:,0], X[:,1], s=70, c=cluster_assignments, alpha=0.8)

# Plot trajectory of centers during the iterations
styles = ['o-r', 'd-g', '+-b', 's-c', '*-y']
for c in range(k):
  axes[1].plot(
    [centers_history[i][c][0] for i in range(n_iters)],
    [centers_history[i][c][1] for i in range(n_iters)],
    styles[c]
  )

# Plot final positions of centers
axes[1].plot(centers[:,0], centers[:,1], '*k', ms=10, zorder=10)
axes[1].set_title(f'Clusters [n_iters={n_iters}]');